<a href="https://colab.research.google.com/github/psvprasad2003/SAMPLE_ML_MODELS/blob/main/Madhav_Executed_Satya_Updated_AE_training_fault_model_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SATYA ECFR Future-Failure XGBoost Models (Updated)

This notebook adapts the original SATYA fault-classification workflow to the ECFR dataset currently available in production. The available panel does not contain the original `Incident Type`, raw sensor values, or raw engine status-word columns. It instead contains historical event-count, rate, recency, and exposure features plus future-failure labels.

Accordingly, this implementation trains three leakage-safe XGBoost binary classifiers:

- Failure within 7 days
- Failure within 14 days
- Failure within 30 days

It preserves the original workflow pattern: engine-level splitting, class-imbalance handling, XGBoost training, held-out evaluation, feature importance, visualizations, and saved model artifacts.


In [ ]:
# CELL 01 - Imports and production configuration
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

ROOT = Path('/raid3/e296408/All_ECFRs_working')
BRANCH_A_DIR = ROOT / 'branch_a_ecfr_only'
PANEL_FILE = BRANCH_A_DIR / 'failure_prediction_panel' / 'ecfr_failure_prediction_panel.parquet'
OUTPUT_DIR = BRANCH_A_DIR / 'failure_prediction_panel' / 'satya_xgboost_outputs'

HORIZONS = [7, 14, 30]
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
NEG_SUBSAMPLE_PER_POS = 20
DAYS_SINCE_LAST_SENTINEL = 9999.0
RANDOM_SEED = 42
TARGET_RECALL = 0.80

PARAM_GRID = [
    {'n_estimators':300, 'max_depth':4, 'learning_rate':0.05, 'min_child_weight':5, 'subsample':0.8, 'colsample_bytree':0.8, 'reg_lambda':1.0},
    {'n_estimators':500, 'max_depth':6, 'learning_rate':0.05, 'min_child_weight':5, 'subsample':0.8, 'colsample_bytree':0.8, 'reg_lambda':1.0},
    {'n_estimators':300, 'max_depth':6, 'learning_rate':0.10, 'min_child_weight':10, 'subsample':0.8, 'colsample_bytree':0.8, 'reg_lambda':2.0},
]

def banner(title):
    print('=' * 100)
    print(title)
    print('=' * 100)

if not PANEL_FILE.is_file():
    raise FileNotFoundError(f'Panel file not found: {PANEL_FILE}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

banner('SATYA XGBOOST FAILURE-PREDICTION SETUP')
print('Panel:', PANEL_FILE)
print('Output directory:', OUTPUT_DIR)
print('Horizons:', HORIZONS)


SATYA XGBOOST FAILURE-PREDICTION SETUP
Panel: /raid3/e296408/All_ECFRs_working/branch_a_ecfr_only/failure_prediction_panel/ecfr_failure_prediction_panel.parquet
Output directory: /raid3/e296408/All_ECFRs_working/branch_a_ecfr_only/failure_prediction_panel/satya_xgboost_outputs
Horizons: [7, 14, 30]


In [ ]:
# CELL 02 - Load and validate the known ECFR panel
def load_and_validate_panel(panel_file):
    panel = pd.read_parquet(panel_file)
    if panel.empty:
        raise ValueError('Panel is empty.')

    labels = [f'FAILS_WITHIN_{h}D' for h in HORIZONS]
    required = ['ENGINE_SERIAL', 'SNAPSHOT_TIME', 'ENGINE_EVER_FAILS_IN_RECORD'] + labels
    missing = [c for c in required if c not in panel.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

    panel['SNAPSHOT_TIME'] = pd.to_datetime(panel['SNAPSHOT_TIME'], errors='coerce')
    if panel['ENGINE_SERIAL'].isna().any():
        raise ValueError('ENGINE_SERIAL contains missing values.')
    if panel['SNAPSHOT_TIME'].isna().any():
        raise ValueError('SNAPSHOT_TIME contains invalid or missing values.')

    for col in labels + ['ENGINE_EVER_FAILS_IN_RECORD']:
        values = set(panel[col].dropna().unique().tolist())
        if panel[col].isna().any() or not values.issubset({0, 1, False, True}):
            raise ValueError(f'{col} must be complete and binary 0/1.')
        panel[col] = panel[col].astype(np.int8)

    bad = (panel[labels[0]] > panel[labels[1]]) | (panel[labels[1]] > panel[labels[2]])
    if bad.any():
        raise ValueError(f'{int(bad.sum()):,} rows violate 7D <= 14D <= 30D label nesting.')

    excluded = set(required)
    feature_cols = [c for c in panel.columns if c not in excluded]
    non_numeric = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(panel[c])]
    if non_numeric:
        raise ValueError(f'Non-numeric features found: {non_numeric[:30]}')

    panel = panel.sort_values(['ENGINE_SERIAL', 'SNAPSHOT_TIME']).reset_index(drop=True)
    schema = pd.DataFrame({
        'column': panel.columns,
        'dtype': panel.dtypes.astype(str).values,
        'missing_count': panel.isna().sum().values,
        'missing_pct': (panel.isna().mean() * 100).round(4).values,
    })

    banner('SCHEMA VERIFIED')
    print(f'Rows={len(panel):,}; columns={panel.shape[1]:,}; engines={panel.ENGINE_SERIAL.nunique():,}; features={len(feature_cols):,}')
    print('Date range:', panel.SNAPSHOT_TIME.min(), 'to', panel.SNAPSHOT_TIME.max())
    for col in labels:
        print(f'{col}: {int(panel[col].sum()):,} positives ({panel[col].mean():.6%})')
    display(schema.head(35))
    return panel, labels, feature_cols, schema


In [ ]:
# CELL 03 - Prepare features and engine-level split
def prepare_features_and_split(panel, labels, feature_cols):
    panel = panel.copy()
    days_cols = [c for c in feature_cols if c.endswith('__DAYS_SINCE_LAST')]
    other_cols = [c for c in feature_cols if c not in days_cols]
    panel[other_cols] = panel[other_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if days_cols:
        panel[days_cols] = panel[days_cols].replace([np.inf, -np.inf], np.nan).fillna(DAYS_SINCE_LAST_SENTINEL)

    X = panel[feature_cols].to_numpy(dtype=np.float32)
    Y = panel[labels].to_numpy(dtype=np.int8)
    if not np.isfinite(X).all():
        raise ValueError('Feature matrix contains non-finite values after imputation.')

    ever = panel.groupby('ENGINE_SERIAL')['ENGINE_EVER_FAILS_IN_RECORD'].first().astype(int).to_dict()
    engines = np.array(list(ever), dtype=object)
    strata = np.array([ever[e] for e in engines])
    rng = np.random.RandomState(RANDOM_SEED)
    splits = {'train': [], 'val': [], 'test': []}

    for label in [0, 1]:
        pool = engines[strata == label].copy()
        rng.shuffle(pool)
        train_end = int(len(pool) * TRAIN_FRAC)
        val_end = train_end + int(len(pool) * VAL_FRAC)
        splits['train'] += list(pool[:train_end])
        splits['val'] += list(pool[train_end:val_end])
        splits['test'] += list(pool[val_end:])

    splits = {name: set(values) for name, values in splits.items()}
    assert not (splits['train'] & splits['val'])
    assert not (splits['train'] & splits['test'])
    assert not (splits['val'] & splits['test'])

    row_split = np.where(
        panel['ENGINE_SERIAL'].isin(splits['train']), 'train',
        np.where(panel['ENGINE_SERIAL'].isin(splits['val']), 'val', 'test')
    )

    banner('ENGINE-LEVEL, FAILURE-STRATIFIED SPLIT')
    for name in ['train', 'val', 'test']:
        idx = np.where(row_split == name)[0]
        print(f'{name:>5}: {len(splits[name]):,} engines; {len(idx):,} rows; ever-fail engines={sum(ever[e] for e in splits[name]):,}')
    return panel, X, Y, days_cols, splits, row_split


In [ ]:
# CELL 04 - Training, tuning, and threshold helpers
def training_sample(y, row_split, seed):
    train_rows = np.where(row_split == 'train')[0]
    positives = train_rows[y[train_rows] == 1]
    negatives = train_rows[y[train_rows] == 0]
    if len(positives) == 0:
        raise ValueError('No positive rows in the training split.')
    rng = np.random.RandomState(seed)
    keep_n = min(len(negatives), len(positives) * NEG_SUBSAMPLE_PER_POS)
    kept_negatives = rng.choice(negatives, size=keep_n, replace=False)
    sample = np.concatenate([positives, kept_negatives])
    rng.shuffle(sample)
    return sample

def binary_metrics(y_true, probability):
    if np.unique(y_true).size < 2:
        return {'roc_auc': np.nan, 'pr_auc': np.nan}
    return {
        'roc_auc': roc_auc_score(y_true, probability),
        'pr_auc': average_precision_score(y_true, probability),
    }

def select_threshold(y_true, probability, target_recall=TARGET_RECALL):
    if y_true.sum() == 0:
        return 0.5
    precision, recall, thresholds = precision_recall_curve(y_true, probability)
    candidates = np.where(recall[:-1] >= target_recall)[0]
    if len(candidates) == 0:
        return 0.5
    best = candidates[np.argmax(precision[:-1][candidates])]
    return float(thresholds[best])

def tune_model(X, y, train_idx, val_idx):
    negatives = int((y[train_idx] == 0).sum())
    positives = int((y[train_idx] == 1).sum())
    scale_pos_weight = negatives / max(positives, 1)
    rows = []
    best_model = None
    best_params = None
    best_score = -np.inf

    for params in PARAM_GRID:
        model = xgb.XGBClassifier(
            objective='binary:logistic',
            eval_metric='aucpr',
            tree_method='hist',
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_SEED,
            n_jobs=-1,
            **params,
        )
        model.fit(X[train_idx], y[train_idx], verbose=False)
        probability = model.predict_proba(X[val_idx])[:, 1]
        metrics = binary_metrics(y[val_idx], probability)
        rows.append({**params, 'scale_pos_weight': scale_pos_weight, **metrics})
        score = -np.inf if np.isnan(metrics['pr_auc']) else metrics['pr_auc']
        if score > best_score:
            best_score = score
            best_model = model
            best_params = params.copy()

    if best_model is None:
        raise RuntimeError('Model selection failed.')
    return best_model, best_params, pd.DataFrame(rows)


In [ ]:
# CELL 05 - Visualizations
def visualize_results(horizon, y_test, probability, predicted, importances, cm, output_dir):
    horizon_dir = Path(output_dir) / f'{horizon}d_visualizations'
    horizon_dir.mkdir(parents=True, exist_ok=True)

    fpr, tpr, _ = roc_curve(y_test, probability)
    plt.figure(figsize=(10, 8))
    plt.plot(fpr, tpr, lw=2.5, label=f'ROC AUC = {roc_auc_score(y_test, probability):.4f}')
    plt.plot([0, 1], [0, 1], linestyle='--', label='Random classifier')
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - Failure Within {horizon} Days'); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(horizon_dir / '01_roc_curve.png', dpi=300); plt.close()

    precision, recall, _ = precision_recall_curve(y_test, probability)
    plt.figure(figsize=(10, 8))
    plt.plot(recall, precision, lw=2.5, label=f'PR AUC = {average_precision_score(y_test, probability):.4f}')
    plt.xlabel('Recall'); plt.ylabel('Precision')
    plt.title(f'Precision-Recall Curve - Failure Within {horizon} Days'); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(horizon_dir / '02_precision_recall_curve.png', dpi=300); plt.close()

    plt.figure(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No failure', 'Failure'], yticklabels=['No failure', 'Failure'])
    plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title(f'Confusion Matrix - {horizon}D')
    plt.tight_layout(); plt.savefig(horizon_dir / '03_confusion_matrix.png', dpi=300); plt.close()

    top = importances.head(20).sort_values()
    plt.figure(figsize=(12, 10))
    plt.barh(top.index, top.values)
    plt.xlabel('XGBoost Importance'); plt.title(f'Top 20 Features - {horizon}D')
    plt.tight_layout(); plt.savefig(horizon_dir / '04_feature_importance.png', dpi=300); plt.close()

    plt.figure(figsize=(12, 7))
    plt.hist(probability[y_test == 0], bins=50, alpha=0.7, label='No failure')
    plt.hist(probability[y_test == 1], bins=50, alpha=0.7, label='Failure')
    plt.xlabel('Predicted Probability'); plt.ylabel('Frequency')
    plt.title(f'Prediction Probability Distribution - {horizon}D'); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(horizon_dir / '05_probability_distribution.png', dpi=300); plt.close()


In [ ]:
# CELL 06 - Complete pipeline
def run_pipeline(panel_file=PANEL_FILE, output_dir=OUTPUT_DIR):
    started = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    panel, labels, feature_cols, schema = load_and_validate_panel(panel_file)
    panel, X, Y, days_cols, splits, row_split = prepare_features_and_split(panel, labels, feature_cols)
    val_idx = np.where(row_split == 'val')[0]
    test_idx = np.where(row_split == 'test')[0]

    models = {}
    tuning_frames = []
    performance_rows = []
    threshold_rows = []
    predictions = pd.DataFrame({
        'ENGINE_SERIAL': panel.loc[test_idx, 'ENGINE_SERIAL'].values,
        'SNAPSHOT_TIME': panel.loc[test_idx, 'SNAPSHOT_TIME'].values,
    })

    for column_idx, horizon in enumerate(HORIZONS):
        y = Y[:, column_idx]
        train_idx = training_sample(y, row_split, RANDOM_SEED + horizon)
        banner(f'XGBOOST FAILURE MODEL: {horizon}-DAY HORIZON')
        print(f'Train sample={len(train_idx):,}, positives={int(y[train_idx].sum()):,}')
        print(f'Validation positives={int(y[val_idx].sum()):,}; test positives={int(y[test_idx].sum()):,}')

        model, best_params, tuning = tune_model(X, y, train_idx, val_idx)
        tuning.insert(0, 'HORIZON_DAYS', horizon)
        tuning_frames.append(tuning)
        models[horizon] = model

        val_probability = model.predict_proba(X[val_idx])[:, 1]
        test_probability = model.predict_proba(X[test_idx])[:, 1]
        val_metrics = binary_metrics(y[val_idx], val_probability)
        test_metrics = binary_metrics(y[test_idx], test_probability)
        threshold = select_threshold(y[val_idx], val_probability)
        predicted = (test_probability >= threshold).astype(np.int8)
        cm = confusion_matrix(y[test_idx], predicted, labels=[0, 1])

        print('Best parameters:', best_params)
        print('Validation:', val_metrics)
        print('Test:', test_metrics)
        print('Validation-derived threshold:', threshold)
        print(classification_report(y[test_idx], predicted, target_names=['No failure', 'Failure'], zero_division=0))

        importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
        importances.rename('importance').to_csv(output_dir / f'feature_importance_{horizon}d.csv', header=True)
        visualize_results(horizon, y[test_idx], test_probability, predicted, importances, cm, output_dir)
        joblib.dump(model, output_dir / f'satya_xgboost_{horizon}d.joblib')

        performance_rows.append({
            'HORIZON_DAYS': horizon,
            'VAL_ROC_AUC': val_metrics['roc_auc'],
            'VAL_PR_AUC': val_metrics['pr_auc'],
            'TEST_ROC_AUC': test_metrics['roc_auc'],
            'TEST_PR_AUC': test_metrics['pr_auc'],
            **{f'BEST_{key.upper()}': value for key, value in best_params.items()},
        })
        threshold_rows.append({
            'HORIZON_DAYS': horizon,
            'VALIDATION_THRESHOLD_TARGET_RECALL': threshold,
            'TEST_CONFUSION_MATRIX': cm.tolist(),
        })
        predictions[f'LABEL_{horizon}D'] = y[test_idx]
        predictions[f'PRED_PROB_{horizon}D'] = test_probability
        predictions[f'PRED_CLASS_{horizon}D'] = predicted

    performance = pd.DataFrame(performance_rows)
    thresholds = pd.DataFrame(threshold_rows)
    tuning_results = pd.concat(tuning_frames, ignore_index=True)

    schema.to_csv(output_dir / 'verified_schema.csv', index=False)
    performance.to_csv(output_dir / 'test_performance_summary.csv', index=False)
    thresholds.to_csv(output_dir / 'threshold_sensitivity.csv', index=False)
    tuning_results.to_csv(output_dir / 'hyperparameter_tuning.csv', index=False)
    predictions.to_csv(output_dir / 'test_predictions.csv', index=False)
    joblib.dump({'models': models, 'feature_cols': feature_cols, 'horizons': HORIZONS}, output_dir / 'satya_xgboost_bundle.joblib')

    config = {
        'panel_file': str(panel_file),
        'output_dir': str(output_dir),
        'feature_cols': feature_cols,
        'days_since_cols': days_cols,
        'horizons': HORIZONS,
        'seed': RANDOM_SEED,
        'negative_subsample_per_positive': NEG_SUBSAMPLE_PER_POS,
        'target_recall': TARGET_RECALL,
        'adaptation_note': 'Uses available future-failure panel labels and historical aggregate features; raw status words, raw sensors, and Incident Type are not present in this panel.',
    }
    (output_dir / 'model_config.json').write_text(json.dumps(config, indent=2))

    banner('PIPELINE COMPLETE')
    display(performance)
    print('Artifacts:', output_dir)
    for path in sorted(output_dir.iterdir()):
        print(' -', path.name)
    print(f'Elapsed: {time.time() - started:.1f}s')
    return {'models': models, 'performance': performance, 'thresholds': thresholds, 'predictions': predictions, 'output_dir': output_dir}


## Run the complete pipeline

The notebook is configured for the known production panel. Run all cells in order.


In [ ]:
# CELL 07 - Run
results = run_pipeline(PANEL_FILE, OUTPUT_DIR)


SCHEMA VERIFIED
Rows=1,277,405; columns=35; engines=1,621; features=29
Date range: 1969-02-09 20:16:54 to 2068-05-05 16:57:12
FAILS_WITHIN_7D: 188 positives (0.014717%)
FAILS_WITHIN_14D: 334 positives (0.026147%)
FAILS_WITHIN_30D: 609 positives (0.047675%)


,column,dtype,missing_count,missing_pct
0,ENGINE_SERIAL,object,0,0.0000
1,SNAPSHOT_TIME,datetime64[ns],0,0.0000
2,chip__COUNT_LOOKBACK,int64,0,0.0000
3,chip__RATE_PER_DAY,float64,0,0.0000
4,chip__DAYS_SINCE_LAST,float64,1238841,96.9811
5,clm_performance__COUNT_LOOKBACK,int64,0,0.0000
6,clm_performance__RATE_PER_DAY,float64,0,0.0000
7,clm_performance__DAYS_SINCE_LAST,float64,744325,58.2685
8,cru_performance__COUNT_LOOKBACK,int64,0,0.0000
9,cru_performance__RATE_PER_DAY,float64,0,0.0000


ENGINE-LEVEL, FAILURE-STRATIFIED SPLIT
train: 1,134 engines; 906,113 rows; ever-fail engines=131
  val: 242 engines; 185,851 rows; ever-fail engines=28
 test: 245 engines; 185,441 rows; ever-fail engines=29
XGBOOST FAILURE MODEL: 7-DAY HORIZON
Train sample=2,751, positives=131
Validation positives=28; test positives=29
Best parameters: {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0}
Validation: {'roc_auc': 0.9731394737388345, 'pr_auc': 0.0886780570850504}
Test: {'roc_auc': 0.9836629441088142, 'pr_auc': 0.08649084031694983}
Validation-derived threshold: 0.874581515789032
              precision    recall  f1-score   support

  No failure       1.00      0.99      1.00    185412
     Failure       0.02      0.86      0.04        29

    accuracy                           0.99    185441
   macro avg       0.51      0.93      0.52    185441
weighted avg       1.00      0.99      1.00    185441

,HORIZON_DAYS,VAL_ROC_AUC,VAL_PR_AUC,TEST_ROC_AUC,TEST_PR_AUC,BEST_N_ESTIMATORS,BEST_MAX_DEPTH,BEST_LEARNING_RATE,BEST_MIN_CHILD_WEIGHT,BEST_SUBSAMPLE,BEST_COLSAMPLE_BYTREE,BEST_REG_LAMBDA
0,7,0.973139,0.088678,0.983663,0.086491,300,4,0.05,5,0.8,0.8,1.0
1,14,0.969994,0.106266,0.966349,0.121401,500,6,0.05,5,0.8,0.8,1.0
2,30,0.935628,0.105880,0.944324,0.183697,300,4,0.05,5,0.8,0.8,1.0


Artifacts: /raid3/e296408/All_ECFRs_working/branch_a_ecfr_only/failure_prediction_panel/satya_xgboost_outputs
 - 14d_visualizations
 - 30d_visualizations
 - 7d_visualizations
 - feature_importance_14d.csv
 - feature_importance_30d.csv
 - feature_importance_7d.csv
 - hyperparameter_tuning.csv
 - model_config.json
 - satya_xgboost_14d.joblib
 - satya_xgboost_30d.joblib
 - satya_xgboost_7d.joblib
 - satya_xgboost_bundle.joblib
 - test_performance_summary.csv
 - test_predictions.csv
 - threshold_sensitivity.csv
 - verified_schema.csv
Elapsed: 818.4s
